In [1]:
#direct SQL client access
from adbc_driver_flightsql import dbapi
from hereutil import here
import yaml

with open(here("db_secret.yaml"), "r") as f:
    db_params = yaml.safe_load(f)

with dbapi.connect(db_params["uri"], db_kwargs=dict(username=db_params["username"], password=db_params["password"])) as con:
    with con.cursor() as cur:
        cur.execute("SELECT * FROM books.hpb LIMIT 10")
        print(cur.fetch_arrow_table())


pyarrow.Table
record_number: int32
field_number: int32
subfield_number: int32
subfield_code: string
value: string
field_code: string
----
record_number: [[1,2,3,4,5,6,7,8,9,10]]
field_number: [[1,1,1,1,1,1,1,1,1,1]]
subfield_number: [[1,1,1,1,1,1,1,1,1,1]]
subfield_code: [["0","0","0","0","0","0","0","0","0","0"]]
value: [["9831:07-11-14","9831:06-11-14","9831:06-11-14","9831:05-11-14","9831:13-10-14","9831:13-10-14","9831:01-10-14","9831:25-09-14","9831:25-09-14","9831:09-09-14"]]
field_code: [["001A","001A","001A","001A","001A","001A","001A","001A","001A","001A"]]


In [2]:

from sqlframe_gizmosql import GizmoSQLSession, GizmoSQLTable
import sqlframe_gizmosql.functions as F

from hereutil import here
import yaml

with open(here("db_secret.yaml"), "r") as f:
    db_params = yaml.safe_load(f)


# Create an SQLFrame (PySpark) session connected to GizmoSQL
session: GizmoSQLSession = GizmoSQLSession.builder \
    .config("gizmosql.uri", db_params["uri"]) \
    .config("gizmosql.username", db_params["username"]) \
    .config("gizmosql.password", db_params["password"]) \
    .getOrCreate()

c = F.col

p_year_of_publication: GizmoSQLTable = session.table("books.all_years_of_publication")

(p_year_of_publication
    .filter(c("year_of_publication").between(1400, 2030))
    .groupBy("year_of_publication")
    .count()
    .orderBy("count", ascending=False)
).show()


+---------------------+---------+
| year_of_publication |  count  |
+---------------------+---------+
|         2021        | 2826262 |
|         2022        | 2742139 |
|         2023        | 2595778 |
|         2024        | 2561238 |
|         2018        | 2496334 |
|         2020        | 2470297 |
|         2019        | 2437548 |
|         2017        | 2386417 |
|         2016        | 2318981 |
|         2015        | 2046770 |
|         2014        | 1910370 |
|         2013        | 1782621 |
|         2006        | 1606442 |
|         2012        | 1602766 |
|         2011        | 1571646 |
|         2009        | 1541896 |
|         2010        | 1480150 |
|         2008        | 1468370 |
|         2007        | 1359864 |
|         2004        | 1315108 |
+---------------------+---------+


In [3]:
import narwhals as nw
c = nw.col

from sqlframe_gizmosql import GizmoSQLSession

from hereutil import here
import yaml

with open(here("db_secret.yaml"), "r") as f:
    db_params = yaml.safe_load(f)


# Create an SQLFrame (PySpark) session connected to GizmoSQL
session: GizmoSQLSession = GizmoSQLSession.builder \
    .config("gizmosql.uri", db_params["uri"]) \
    .config("gizmosql.username", db_params["username"]) \
    .config("gizmosql.password", db_params["password"]) \
    .getOrCreate()

# Turn the SQLFrame table into a Narwhal LazyFrame (nice ~polars API)
p_year_of_publication: nw.LazyFrame = nw.from_native(session.table("books.all_years_of_publication"))

(p_year_of_publication
    .filter((c("year_of_publication") >= 1400) & (c("year_of_publication") <= 2030))
    .group_by(c("year_of_publication"))
    .agg(nw.len())
    .sort('len', descending=True)
    .collect(backend='polars')
)

┌─────────────────────────────────┐
|       Narwhals DataFrame        |
|---------------------------------|
|shape: (631, 2)                  |
|┌─────────────────────┬─────────┐|
|│ year_of_publication ┆ len     │|
|│ ---                 ┆ ---     │|
|│ i32                 ┆ i64     │|
|╞═════════════════════╪═════════╡|
|│ 2021                ┆ 2826262 │|
|│ 2022                ┆ 2742139 │|
|│ 2023                ┆ 2595778 │|
|│ 2024                ┆ 2561238 │|
|│ 2018                ┆ 2496334 │|
|│ …                   ┆ …       │|
|│ 1408                ┆ 6       │|
|│ 1403                ┆ 5       │|
|│ 1411                ┆ 4       │|
|│ 1407                ┆ 4       │|
|│ 1438                ┆ 4       │|
|└─────────────────────┴─────────┘|
└─────────────────────────────────┘

In [4]:
# Use the provided common basis, which registers all the tables as narwhals lazyframes. 

import hereutil
hereutil.add_to_sys_path(hereutil.here())
from src.common_basis_gizmosql import *

p(f('all_years_of_publication')
    .filter((c("year_of_publication") >= 1400) & (c("year_of_publication") <= 2030))
    .group_by(c("year_of_publication"))
    .agg(nw.len())
    .sort('len', descending=True)
)

year_of_publication,len
i32,i64
2021,2826262
2022,2742139
2023,2595778
2024,2561238
2018,2496334
…,…
1408,6
1403,5
1407,4
